# Pipeline 3 - Next Best Channel Predictor

Generated: 2026-04-07T11:53:52.837536Z


## 1) Problem Framing

**Business question:** Recommend the next donation channel most likely to work for each supporter.

**Who cares:** nonprofit leadership, program staff, and/or fundraising staff depending on the pipeline domain.

**Why it matters:** this model turns operational, donor, or outreach data into a decision-support signal for a resource-constrained safehouse nonprofit.

**Predictive vs. explanatory goal:** this notebook includes both. The predictive model is evaluated on unseen data and is used for deployment-oriented scoring. The explanatory or relationship model is included to identify which variables appear most connected to the target and to support business interpretation. We do not treat predictive accuracy as causal proof.

**Success metrics:** classification pipelines use accuracy, F1, and ROC AUC where appropriate. Regression/forecasting pipelines use MAE, RMSE, and R-squared. The notebook also compares against a simple baseline so the results can be interpreted honestly.


## Notebook Setup

Shared imports and helper functions are defined once here so the later rubric sections can focus on the pipeline-specific code for this business problem.


In [1]:
import warnings

warnings.filterwarnings("ignore")

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Ensure the shared pipeline modules are importable when running from anywhere.
# (Repo-relative, so it works after pushing to GitHub.)
cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "ml-pipelines" else cwd
if (repo_root / "ml-pipelines").exists():
    ml_pipelines_dir = (repo_root / "ml-pipelines").resolve()
    if str(ml_pipelines_dir) not in sys.path:
        sys.path.insert(0, str(ml_pipelines_dir))

from data_prep import (
    as_bool,
    compare_profiles,
    dataset_profile,
    export_predictions_json,
    fill_numeric_median,
    get_project_paths,
    load_df,
    numeric,
    prep,
    quick_eda,
    time_split,
)
from modeling import (
    CandidateResult,
    ablate_feature_groups_one_by_one_cv,
    cv_evaluate_candidate,
    cv_results_table,
    evaluate_candidate,
    read_json_if_exists,
    results_table,
    select_simplest_within_delta,
    select_simplest_within_delta_cv,
    should_export_by_guardrail,
    top_features,
    tune_model_randomized,
    write_ablation_report_json,
    write_run_report_json,
)

paths = get_project_paths()
print("Data dir:", paths.data_dir)
print("Output dir:", paths.out_dir)


def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)


Data dir: /Users/xenorex/Downloads/lighthouse_csv_v7
Output dir: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions


## 2) Data Acquisition, Preparation, and Exploration

The pipeline reads the provided CSV files from `data/raw/`, performs joins and feature engineering in code, handles missing values reproducibly, and prints an EDA summary before modeling. The EDA step checks row counts, missingness, target distributions, feature summaries, and key categorical distributions.


In [2]:
supporters = load_df("supporters")
donations = load_df("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
donations = donations.dropna(subset=["donation_date","supporter_id","channel_source"]).sort_values(["supporter_id","donation_date"]).copy()
donations["amount"] = numeric(donations["amount"])
donations["estimated_value"] = numeric(donations["estimated_value"])
donations["is_recurring_bool"] = as_bool(donations["is_recurring"])
donations["next_channel"] = donations.groupby("supporter_id")["channel_source"].shift(-1)
donations["next_date"] = donations.groupby("supporter_id")["donation_date"].shift(-1)
donations["days_to_next"] = (donations["next_date"] - donations["donation_date"]).dt.days
df = donations[(donations["next_channel"].notna()) & (donations["days_to_next"].between(1, 365))].copy()
df = df.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
df["donations_so_far"] = df.groupby("supporter_id").cumcount() + 1
df["prev_date"] = df.groupby("supporter_id")["donation_date"].shift(1)
df["recency_days"] = (df["donation_date"] - df["prev_date"]).dt.days.fillna(9999).clip(lower=0)
channel_values = ["Campaign", "Direct", "Event", "PartnerReferral", "SocialMedia"]
for channel in channel_values:
    current = df["channel_source"].eq(channel).astype(int)
    df[f"prior_{channel}"] = current.groupby(df["supporter_id"]).cumsum() - current
cat_cols = ["supporter_type","relationship_type","region","country","acquisition_channel","donation_type","campaign_name","channel_source","impact_unit"]
num_cols = ["amount","estimated_value","donations_so_far","recency_days","is_recurring_bool"] + [f"prior_{c}" for c in channel_values]
df[cat_cols] = df[cat_cols].fillna("Unknown")
fill_numeric_median(df, num_cols)
train, test = time_split(df, "donation_date")
print("Rows:", len(df), "Classes:", sorted(df["next_channel"].unique()))
quick_eda(df, "Next-channel modeling table", target_col="next_channel", numeric_cols=num_cols, categorical_cols=cat_cols)


Rows: 339 Classes: ['Campaign', 'Direct', 'Event', 'PartnerReferral', 'SocialMedia']

EDA: Next-channel modeling table
Shape: (339, 30)

Top missing-value rates:
referral_post_id         0.820059
currency_code            0.433628
prev_date                0.168142
donation_id              0.000000
days_to_next             0.000000
prior_PartnerReferral    0.000000
prior_Event              0.000000
prior_Direct             0.000000
prior_Campaign           0.000000
recency_days             0.000000

Target distribution / summary: next_channel
next_channel
Campaign           98
Event              74
Direct             64
SocialMedia        61
PartnerReferral    42
Saved EDA plot: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/eda-plots/next_channel_modeling_table_next_channel_distribution.png

Numeric feature summary:
                           mean       std  min     50%      max
amount                  575.978   770.688  0.0  338.91  6481.54
estimated_value         700.572   

## 3) Modeling and Feature Selection

The feature set is selected from fields that would be available at the decision point whenever possible. Predictive models use ensembles or tuned tree-based models when they improve out-of-sample performance. Explanatory models use simpler linear or logistic models when interpretability matters.


In [3]:
features = cat_cols + num_cols

# Keep an interpretable baseline for explanation.
explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])

# Candidate pool for resource-aware selection.
candidates = {
    "LogisticRegression": explanatory,
    "RandomForestSmall": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestClassifier(n_estimators=200, random_state=42, min_samples_leaf=3))]),
    "GradientBoosting": Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))]),
}



## 4) Evaluation and Selection

The notebook uses a train/test or time-based holdout split depending on the business problem. Where appropriate, it also uses compact cross-validation, model comparison tables, and lightweight randomized tuning. Metrics are interpreted in business terms rather than treated as abstract statistics.


In [4]:
# Strict holdout discipline: CV on train, single touch on time-ordered holdout.
X_train, y_train = train[features], train["next_channel"]
X_holdout, y_holdout = test[features], test["next_channel"]

current_profile = dataset_profile(train[features], categorical_cols=cat_cols, numeric_cols=num_cols)
report_path = paths.reports_dir / "next-best-campaign_run_report.json"
prev_report = read_json_if_exists(report_path)
drift = compare_profiles(current_profile, (prev_report or {}).get("data_profile"))

# CV tournament
cv_results = [cv_evaluate_candidate(name, est, X_train, y_train, task="multiclass", topk=2) for name, est in candidates.items()]
cv_tbl = cv_results_table(cv_results).sort_values(by=["top2_accuracy_mean", "accuracy_mean", "fit_s_mean"], ascending=[False, False, True])
print("CV tournament (train only):")
print(cv_tbl.to_string(index=False))

selected_cv = select_simplest_within_delta_cv(cv_results, primary_metric="top2_accuracy", delta=0.01, higher_is_better=True)
base_selected = selected_cv.estimator
base_selected_name = selected_cv.name

# Tune selected family (if applicable)
param_space = {}
if base_selected_name.startswith("RandomForest"):
    param_space = {
        "model__n_estimators": [100, 150, 200, 300],
        "model__max_depth": [None, 3, 5, 8],
        "model__min_samples_leaf": [1, 3, 5, 8],
    }
elif base_selected_name.startswith("GradientBoosting"):
    param_space = {
        "model__n_estimators": [100, 150, 250],
        "model__learning_rate": [0.03, 0.05, 0.1],
        "model__max_depth": [2, 3, 4],
    }

if param_space:
    tune = tune_model_randomized(
        base_selected_name,
        base_selected,
        X_train,
        y_train,
        param_distributions=param_space,
        task="multiclass",
        cv_metric="accuracy",
        n_iter=10,
    )
    tuned_estimator = tune.best_estimator
    tuned_name = f"{base_selected_name}+tuned"
    tuned_params = tune.best_params
else:
    tuned_estimator = base_selected
    tuned_name = base_selected_name
    tuned_params = {}

post_cv = cv_evaluate_candidate(tuned_name, tuned_estimator, X_train, y_train, task="multiclass", topk=2)

# Ablation (CV)
feature_groups = [[c] for c in features]
ablation_tbl = ablate_feature_groups_one_by_one_cv(
    base_name=tuned_name,
    estimator=tuned_estimator,
    X=X_train,
    y=y_train,
    task="multiclass",
    feature_groups=feature_groups,
    primary_metric="top2_accuracy",
    higher_is_better=True,
    topk=2,
)
write_ablation_report_json(paths.reports_dir / "next-best-campaign_ablation_report.json", {"pipeline": "next-best-campaign", "table": ablation_tbl.to_dict(orient="records")})

# Final holdout report
final_model = tuned_estimator
final_model.fit(X_train, y_train)
holdout_pred = final_model.predict(X_holdout)
holdout_proba = final_model.predict_proba(X_holdout)
from sklearn.metrics import top_k_accuracy_score, accuracy_score
classes = final_model.named_steps["model"].classes_
holdout_metrics = {
    "accuracy": float(accuracy_score(y_holdout, holdout_pred)),
    "top2_accuracy": float(top_k_accuracy_score(y_holdout, holdout_proba, k=2, labels=classes)),
}

allow_export, guardrail = should_export_by_guardrail(
    previous_report=prev_report,
    current_holdout={"top2_accuracy": holdout_metrics["top2_accuracy"]},
    primary_metric="top2_accuracy",
    min_delta_allowed=0.01,
    higher_is_better=True,
)

write_run_report_json(
    report_path,
    {
        "pipeline": "next-best-campaign",
        "primary_metric": "top2_accuracy",
        "selected_family": base_selected_name,
        "tuned_name": tuned_name,
        "tuned_params": tuned_params,
        "cv_table": cv_tbl.to_dict(orient="records"),
        "post_tune_cv": post_cv.metrics_mean,
        "holdout": holdout_metrics,
        "guardrail": guardrail,
        "data_profile": current_profile,
        "drift": drift,
    },
)
print("Wrote run report:", report_path)

predictive = final_model
explanatory.fit(X_train, y_train)



CV tournament (train only):
             model  fit_s_mean  predict_s_mean  accuracy_mean  top2_accuracy_mean  accuracy_std  top2_accuracy_std
 RandomForestSmall    0.133761        0.025632       0.295238            0.499953      0.010131           0.025673
LogisticRegression    0.422268        0.017156       0.287302            0.464566      0.028199           0.029413
  GradientBoosting    0.387860        0.039215       0.271615            0.464472      0.042026           0.027263
Wrote run report: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/run-reports/next-best-campaign_run_report.json


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sp

## 5) Causal and Relationship Analysis

The relationship analysis section highlights important features and discusses whether those relationships make sense for the organization. These findings are observational: they can guide hypotheses and strategy, but they are not automatically causal. Any sensitive resident-care decision must remain human-reviewed.


In [5]:
print("Top predictive features:")
print(top_features(predictive).to_string(index=False))
print("Top explanatory relationships:")
print(top_features(explanatory).to_string(index=False))
print_business_takeaway("Treat the next-channel result as a ranked recommendation. Top-2 accuracy is more useful than exact-channel accuracy because outreach teams can test two good options.")


Top predictive features:
                   feature  importance
      num__estimated_value    0.090170
         num__recency_days    0.078506
       num__prior_Campaign    0.066868
               num__amount    0.059494
     num__donations_so_far    0.055797
    num__prior_SocialMedia    0.041623
          num__prior_Event    0.038931
num__prior_PartnerReferral    0.036986
    cat__impact_unit_hours    0.030673
         num__prior_Direct    0.029516
Top explanatory relationships:
                                 feature  importance
              num__prior_PartnerReferral    0.202498
              cat__campaign_name_Unknown    0.183825
          cat__acquisition_channel_Event    0.164537
                        num__prior_Event    0.149199
                       cat__region_Luzon    0.136279
                  cat__impact_unit_hours    0.132322
cat__acquisition_channel_PartnerReferral    0.122282
                  cat__impact_unit_pesos    0.120438
             cat__donation_type_Moneta

## 6) Deployment Notes

The final scoring step exports JSON to `output/ml-predictions/`. These files match the API import contract used by `POST /api/ml/import?replace=true` and can be viewed in the deployed staff portal under `/app/ml` or the ML action center.


In [6]:
latest = donations.groupby("supporter_id").tail(1).copy()
latest = latest.merge(supporters[["supporter_id","supporter_type","relationship_type","region","country","acquisition_channel"]], on="supporter_id", how="left")
history = donations.groupby("supporter_id").agg(
    donations_so_far=("donation_id","count"),
    prev_date=("donation_date", lambda s: s.iloc[-2] if len(s) >= 2 else pd.NaT),
).reset_index()
latest = latest.merge(history, on="supporter_id", how="left")
latest["recency_days"] = (latest["donation_date"] - latest["prev_date"]).dt.days.fillna(9999).clip(lower=0)
latest["is_recurring_bool"] = as_bool(latest["is_recurring"])
channel_counts = donations.pivot_table(index="supporter_id", columns="channel_source", values="donation_id", aggfunc="count", fill_value=0).reset_index()
latest = latest.merge(channel_counts, on="supporter_id", how="left")
for channel in channel_values:
    latest[f"prior_{channel}"] = latest[channel] if channel in latest.columns else 0
latest[cat_cols] = latest[cat_cols].fillna("Unknown")
fill_numeric_median(latest, num_cols)
proba = predictive.predict_proba(latest[features])
classes = predictive.named_steps["model"].classes_
best_idx = proba.argmax(axis=1)
latest["predicted_channel"] = [classes[i] for i in best_idx]
latest["confidence"] = proba.max(axis=1)
export_predictions_json(
    "next_channel_source",
    "Supporter",
    latest[["supporter_id","confidence","predicted_channel","channel_source","campaign_name","acquisition_channel","supporter_type"]],
    "supporter_id",
    "confidence",
    "predicted_channel",
)


Wrote 59 predictions: /Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/next_channel_source.json


PosixPath('/Users/xenorex/RiderProjects/INTEX2/output/ml-predictions/next_channel_source.json')